<a href="https://colab.research.google.com/github/tat5603-prog/Trego_AI_Portfolio/blob/main/Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ------------------------------------------------------------
# STEP 2: Import Libraries
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [ ]:
# ------------------------------------------------------------
# STEP 3: Load the Kaggle Dataset
# ------------------------------------------------------------

file_path = "/content/drive/MyDrive/datasets/student_exam_performance.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

Dataset loaded successfully!
Number of rows: 100000
Number of columns: 44


In [ ]:
# ------------------------------------------------------------
# STEP 4: Look at the Dataset
# ------------------------------------------------------------

print(df.head())

   student_id  age  gender education_level school_type family_income  \
0  STU_000001   20  Female     High School      Public        Middle   
1  STU_000002   17    Male     High School      Public        Middle   
2  STU_000003   18   Other   Undergraduate      Public        Middle   
3  STU_000004   20    Male     High School      Public        Middle   
4  STU_000005   16  Female     High School     Private          High   

  parent_education urban_rural  previous_exam_score  previous_gpa  ...  \
0              NaN       Rural                78.36          2.95  ...   
1              NaN    Suburban                73.40          2.89  ...   
2           Master       Urban                82.76          3.35  ...   
3      High School    Suburban                60.61          2.57  ...   
4         Bachelor    Suburban                74.79          2.84  ...   

   exam_difficulty  exam_preparation_days questions_attempted  \
0           Medium                     27                

In [ ]:
print(df.columns)

Index(['student_id', 'age', 'gender', 'education_level', 'school_type',
       'family_income', 'parent_education', 'urban_rural',
       'previous_exam_score', 'previous_gpa', 'attendance_percentage',
       'assignment_completion_rate', 'class_participation',
       'study_hours_per_day', 'self_study_hours', 'private_tuition',
       'online_learning_hours', 'study_consistency', 'study_environment',
       'study_method', 'revision_frequency', 'practice_tests_completed',
       'notes_quality', 'sleep_hours', 'sleep_quality', 'daily_screen_time',
       'physical_activity_hours', 'break_frequency', 'stress_level',
       'motivation_level', 'internet_access', 'device_availability',
       'educational_app_usage', 'online_course_hours', 'exam_difficulty',
       'exam_preparation_days', 'questions_attempted', 'questions_correct',
       'time_management_score', 'exam_anxiety_level', 'exam_score',
       'performance_grade', 'pass_status', 'performance_level'],
      dtype='object')


In [ ]:
# ------------------------------------------------------------
# STEP 5: Understand the Dataset
# ------------------------------------------------------------

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 44 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   student_id                  100000 non-null  object 
 1   age                         100000 non-null  int64  
 2   gender                      100000 non-null  object 
 3   education_level             100000 non-null  object 
 4   school_type                 100000 non-null  object 
 5   family_income               100000 non-null  object 
 6   parent_education            93474 non-null   object 
 7   urban_rural                 100000 non-null  object 
 8   previous_exam_score         100000 non-null  float64
 9   previous_gpa                92207 non-null   float64
 10  attendance_percentage       90137 non-null   float64
 11  assignment_completion_rate  100000 non-null  float64
 12  class_participation         100000 non-null  object 
 13  study_hours_per

In [ ]:
print(df.describe())

                 age  previous_exam_score  previous_gpa  \
count  100000.000000        100000.000000  92207.000000   
mean       17.005740            69.531033      2.780892   
std         1.999562            11.379108      0.478218   
min        14.000000            30.000000      1.000000   
25%        15.000000            61.820000      2.460000   
50%        17.000000            69.530000      2.780000   
75%        19.000000            77.232500      3.110000   
max        20.000000           100.000000      4.000000   

       attendance_percentage  assignment_completion_rate  study_hours_per_day  \
count           90137.000000               100000.000000        100000.000000   
mean               84.616882                   69.724405             2.995725   
std                 7.922485                   12.127179             1.719466   
min                46.340000                   20.000000             0.500000   
25%                79.210000                   61.480000       

We want to predict:

Survived

0 = Did not survive
1 = Survived

Pclass
Age
SibSp
Parch
Fare

In [ ]:
# ------------------------------------------------------------
# STEP 6: Select Features and Target
# ------------------------------------------------------------

features = [
    "age",
    "previous_exam_score",
    "previous_gpa",
    "attendance_percentage",
    "study_hours_per_day"
]

X = df[features]

y = df["pass_status"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
   age  previous_exam_score  previous_gpa  attendance_percentage  \
0   20                78.36          2.95                  81.49   
1   17                73.40          2.89                    NaN   
2   18                82.76          3.35                 100.00   
3   20                60.61          2.57                  84.91   
4   16                74.79          2.84                  88.21   

   study_hours_per_day  
0                 4.94  
1                 3.42  
2                 1.06  
3                 7.89  
4                 2.63  

Target:
0    Pass
1    Pass
2    Pass
3    Pass
4    Pass
Name: pass_status, dtype: object


In [ ]:
# ------------------------------------------------------------
# STEP 7: Check for Missing Values
# ------------------------------------------------------------

print(X.isnull().sum())

age                         0
previous_exam_score         0
previous_gpa             7793
attendance_percentage    9863
study_hours_per_day         0
dtype: int64


In [ ]:
# ------------------------------------------------------------
# STEP 8: Handle Missing Values
# ------------------------------------------------------------

X = X.copy()

X["previous_gpa"] = X["previous_gpa"].fillna(X["previous_gpa"].median())
X["attendance_percentage"] = X["attendance_percentage"].fillna(X['attendance_percentage'].median())

print("Missing values after cleaning:")
print(X.isnull().sum())

Missing values after cleaning:
age                      0
previous_exam_score      0
previous_gpa             0
attendance_percentage    0
study_hours_per_day      0
dtype: int64


In [ ]:
# ------------------------------------------------------------
# STEP 9: Split Data into Training and Testing Sets
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))

Training data: 80000
Testing data: 20000


In [ ]:
# ------------------------------------------------------------
# STEP 10: Create Logistic Regression Model
# ------------------------------------------------------------

model = LogisticRegression(max_iter=1000)

print(model)

LogisticRegression(max_iter=1000)


In [ ]:
# ------------------------------------------------------------
# STEP 11: Train the Model
# ------------------------------------------------------------

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


fit() is where the machine learning happens.
 The model looks at
 Features + Known Answers
        ↓
   Learn patterns
        ↓
   Logistic Model

In [ ]:
# ------------------------------------------------------------
# STEP 12: Make Predictions
# ------------------------------------------------------------

y_pred = model.predict(X_test)

print("Predictions:")
print(y_pred[:20])

Predictions:
['Fail' 'Pass' 'Fail' 'Pass' 'Fail' 'Pass' 'Pass' 'Pass' 'Fail' 'Pass'
 'Pass' 'Pass' 'Fail' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass' 'Pass']


The predictions will be
0
1
0
1
1
0
... where 0 = Did not survive
1 = Survived

In [ ]:
# ------------------------------------------------------------
# STEP 13: Calculate Accuracy
# ------------------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("Accuracy percentage:", accuracy * 100, "%")

Accuracy: 0.80635
Accuracy percentage: 80.635 %


In [ ]:
# ------------------------------------------------------------
# STEP 14: Confusion Matrix
# ------------------------------------------------------------

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 1428  3036]
 [  837 14699]]


                 ACTUAL
              0          1

PREDICT  0   Correct    Wrong
         1   Wrong      Correct

In [ ]:
# ------------------------------------------------------------
# STEP 15: Classification Report
# ------------------------------------------------------------

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        Fail       0.63      0.32      0.42      4464
        Pass       0.83      0.95      0.88     15536

    accuracy                           0.81     20000
   macro avg       0.73      0.63      0.65     20000
weighted avg       0.78      0.81      0.78     20000



In [ ]:
# ------------------------------------------------------------
# STEP 16: Predict Probabilities
# ------------------------------------------------------------

probabilities = model.predict_proba(X_test)

print(probabilities[:10])

[[0.64799839 0.35200161]
 [0.34283321 0.65716679]
 [0.52713903 0.47286097]
 [0.14608639 0.85391361]
 [0.53366181 0.46633819]
 [0.04967946 0.95032054]
 [0.30725711 0.69274289]
 [0.0677537  0.9322463 ]
 [0.55907172 0.44092828]
 [0.02693822 0.97306178]]


In [ ]:
# ------------------------------------------------------------
# STEP 17: Predict a New Student
# ------------------------------------------------------------

new_student = pd.DataFrame({
    "age": [19],
    "previous_exam_score": [85.3],
    "previous_gpa": [3.8],
    "attendance_percentage": [100],
    "study_hours_per_day": [3]
})

prediction = model.predict(new_student)

probability = model.predict_proba(new_student)

print("Prediction:", prediction)

print("Probability of NOT passing:",
      probability[0][0])

print("Probability of passing:",
      probability[0][1])

Prediction: ['Pass']
Probability of NOT passing: 0.02281508325661108
Probability of passing: 0.9771849167433889


In [ ]:
# ------------------------------------------------------------
# STEP 18: Human-Friendly Prediction
# ------------------------------------------------------------

if prediction[0] == 'Pass':
    print("The model predicts: Pass")
else:
    print("The model predicts: Did not pass")

The model predicts: Pass


In [ ]:
# ------------------------------------------------------------
# STEP 19: Examine Model Coefficients
# ------------------------------------------------------------

coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
})

print(coefficients)

                 Feature  Coefficient
0                    age     0.002710
1    previous_exam_score     0.097896
2           previous_gpa    -0.081734
3  attendance_percentage     0.041158
4    study_hours_per_day     0.476684


https://www.kaggle.com/datasets/mobeenfatimah/student-exam-performance-and-success-dataset

Part 1:

The dataset that I used from kaggle was the Student exam performance and success dataset. The dataset has 44 columns, or features, and 100000 rows. This dataset includes data from 100000 student records and includes information like previous grades, age, and school type. The model will predict whether a student will pass or fail the next exam.

Part 2:

Each row represents a student and some of the important columns are age, previous exam score, previous gpa, attendance percentage, and amount of hours studied. These important columns will be useful in predicting whether the student will pass or fail. The target variable is whether the students pass or fail. 77 percent of students n the dataset passed and 23 percent of the students failed. There are some missing values in some of the columns, like the previous gpa and the attendance status. Some columns should not be used for the prediction like the students id number since it does not affect how the student will do on the exam.

Part 3:

The features used in this model are age, previous exam score, previous gpa, attendance percentage, and amount of hours studied. The target is whether the student will pass or fail. I selected these features because they are some of the most impactful on how someone will perform on an exam.

Part 4:

There are missing values in previous gpa and the attendance percentage of the students. In order to solve the problem of missing values we replace the missing value with the average value from the rest of the data. So the values of the missing gpas become the average gpa from the other students.

Part 5:

The testing data needs to be kept seperate from the training data because if the model already knows the exact correct answer for the data its tested on it will just return the pass and fail value it already knows. Testing it on data it has not seen before prevents this and assures the model is not just returning information we gave it. I used 20 percent testing data and 80 percent training. This means 80000 students data was used for training while 20000 students data were used for testing.

Part 6:

A logostic regression model is one that predicts something from two choices. This is appropriate for my problem because it is predicting whether the student will pass or fail. The pass or fail are the two options that the mpdel will pick from.

Part 7:

The accuracy percentage was 80.635%. The model was able to succesfully identify 1428 true positives and 14699 true negatives. The model also predicted 3036 false positives and 837 false negatives. The percision percentage is the number of true positives over the number of true positives added to the number of false positives, meaning the percision of this model is 0.32. Recall is the number of true positives over the number of true positives plus the number of false negatives. This means the recall for this model was 0.63 percent. The f1-score is .42 and is a combination of the percision and recall score. The metric that is most important for this particular problem is the percision as it would be worse if the model told someone they would pass only for them to fail.

Part 8:

A new observation that the model could predict from this data is whether or not they will get an A grade or higher. This is similar to the model now, just more specific about the score. It would use the same data and information just change the parameters.

Part 9:

My model predicted whether students where going to pass or fail an exam. The model performed fairly well as it scored an 80% in accuracy. I learned from the confusion matrix how many times the model guessed incorrectly and whether it guessed that someone would pass or fail. A first limitation of the model are the missing data that we had add in ourselves. This makes the model less accurate as it does not have the real data to base off of. A second limitation of the model is holding all of the aspects to the same weight. The model is taking into account the persons age just as much as it is taking into account the number of hours the person has studied. This does not make logical sense and would cause the model to be innaccurate. If I had more time I would take into account the fact that the longer a person studied is more impoartant than their age or the grades they have gotten in the past.